# EDA 02 — Q2 Volatilitas per Jam Broker + Q3 Volatilitas per Sesi + Spread

**Rancangan:** EDA v3 Final. **Purpose:** mengukur *bentuk* volatilitas (bukan profit).

## Yang dijawab
- **Q2:** realized volatility (r²) dan mean|r| per **jam dinding broker**, plus
  `tick_volume` sebagai proksi aktivitas, dan rasio sesi terhadap harian.
- **Q3:** volatilitas per **sesi UTC** (label eksklusif primer).
- **Kalibrasi batas sesi M1–M4** (lihat sel masing-masing).
- **Spread terukur:** p50/p95/p99/maks dan %>3 pip per sesi × jam × tahun.

## Peringatan SA-07 dari nb01 (WAJIB dibaca sebelum menafsirkan)
Ekor return M1 didorong **dua sumber**: berita makro (12:30 UTC = 08:30 ET) dan **gap
pembukaan sesi** (21–22 UTC). Volatilitas per jam di bawah **tidak** bisa memisahkan
keduanya karena **kalender berita di luar scope EDA**. Jadi:
- Angka volatilitas per jam/sesi adalah **deskriptif mentah**, bukan "volatilitas bersih".
- Setiap tabel_volatilitas juga melaporkan versi **tanpa return lintas gap** agar efek gap
  bisa dinilai.
- Tidak ada klaim "sesi X lebih volatile karena" tanpa catatan ini.

## Batas keras (Wajib)
- **Tidak** ada P&L, equity, backtest, atau pemilihan metode.
- **Tidak** menulis `configs/`. Fragmen spread **hanya dicetak** (dan disimpan ke
  `reports/eda/2026-10-06/spread_fragment.yaml` sebagai bahan review **4b**).
- **Tidak** ada imputasi.
- Inferensi sinyal **tidak** diambil dari M1.


In [1]:
"""Setup lingkungan dan kunci versi."""
import importlib.metadata
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml


def find_project_root(start: Path) -> Path:
    """Cari root proyek (pyproject.toml + src/) dari cwd ke atas."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("PROJECT_ROOT tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.broker_time import BrokerSchedule, wall_clock_series  # noqa: E402
from src.utils.config import load_config  # noqa: E402

sys.path.insert(0, str(PROJECT_ROOT / "notebooks/scalping"))
from _eda_common import (  # noqa: E402
    CROSS_TAB_HOURS,
    EXPECTED_FINGERPRINT,
    SESSION_ORDER,
    add_labels,
    session_fingerprint,
)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("python       :", sys.version.split()[0])
for package in ("pandas", "numpy", "pyarrow", "matplotlib", "scipy", "pyyaml", "tzdata"):
    print(f"  {package:11s}: {importlib.metadata.version(package)}")

assert importlib.metadata.version("tzdata") == "2026.5", "tzdata harus 2026.5 (Kontrak 00)"
assert (
    int(importlib.metadata.version("pandas").split(".")[0]) == 2
), "Notebook WAJIB jalan di venv kanonik (pandas 2.x); pandas 3.x merusak satuan .asi8."

PROJECT_ROOT : D:\ANN
python       : 3.12.8
  pandas     : 2.2.3
  numpy      : 1.26.4
  pyarrow    : 17.0.0
  matplotlib : 3.10.9
  scipy      : 1.17.1
  pyyaml     : 6.0.2
  tzdata     : 2026.5


## 0. Assertion definisi sesi (anti-drift)

Definisi sesi harus **sama persis** dengan Notebook 00. Modul bersama
`notebooks/scalping/_eda_common.py` menyimpan salinan terkunci; sidik jarinya di-assert
di sini supaya nb02–nb05 tidak diam-diam memakai definisi berbeda (pelajaran SA-03).

In [2]:
"""Kunci definisi sesi agar tidak drift dari Notebook 00."""
print("sidik jari definisi sesi:")
print(" ", session_fingerprint())
assert (
    session_fingerprint() == EXPECTED_FINGERPRINT
), "Definisi sesi berubah vs Notebook 00 -STOP, kembalikan ke kontrak."
print("PASS: definisi sesi identik dengan Notebook 00.")
print("label eksklusif:", " > ".join(SESSION_ORDER))
print("cross-tab UTC-tetap:", CROSS_TAB_HOURS)

sidik jari definisi sesi:
  Tokyo:Asia/Tokyo:9-18|London:Europe/London:8-16|New York:America/New_York:8-17|Sydney:Australia/Sydney:8-16|priority=London>NewYork>Tokyo>Sydney|xtab=is_tokyo:0-9,is_london:8-16,is_ny:13-22,is_sydney:21-6
PASS: definisi sesi identik dengan Notebook 00.
label eksklusif: London > NewYork > Tokyo > Sydney > Off
cross-tab UTC-tetap: {'is_tokyo': (0, 9), 'is_london': (8, 16), 'is_ny': (13, 22), 'is_sydney': (21, 6)}


## 1. Memuat M1 + return + label sesi

M1 adalah sumber spread (`spread_pips`) dan proksi aktivitas (`tick_volume`). Volatilitas
dihitung dari **M1** (bar terkecil) lalu Session & jam; M5/M15 hanya dipakai sebagai
konfirmasi di Q2 (kalibrasi M1–M4 memakai M5 agar tidak terlalu microstructure).

Return dihitung **setelah pengurutan kronologis** (pernah menjadi bug di nb01), dan
sebelum pemisahan label, memakai `shift(1)`.


In [3]:
"""Muat M1 2023-2026, urutkan, hitung return log BID, label broker-day dan sesi."""
config = load_config("data")
SYMBOL = config["mt5"]["symbol"]
SCHEDULE = BrokerSchedule.from_config(config["mt5"])
ANOMALY_WINDOWS = tuple((w["start"], w["end"]) for w in config["validation"]["anomaly_windows"])
YEARS = (2023, 2024, 2025, 2026)
PIP_SIZE = 0.01
COLUMNS = ["time", "broker_time_raw", "close", "tick_volume", "spread_pips"]

frames = [
    pd.read_parquet(
        PROJECT_ROOT / "data/processed/M1" / f"usdjpy_m1_{year}.parquet",
        columns=COLUMNS,
    )
    for year in YEARS
]
m1 = pd.concat(frames, ignore_index=True)
m1["utc"] = pd.to_datetime(m1["time"], utc=True)
m1["wall"] = wall_clock_series(m1["broker_time_raw"])
m1["wall_hour"] = m1["wall"].dt.hour
m1["utc_hour"] = m1["utc"].dt.hour
m1["broker_day"] = m1["wall"].dt.normalize()
m1 = m1.sort_values("utc", kind="mergesort").reset_index(drop=True)
m1["r"] = np.log(m1["close"] / m1["close"].shift(1))
m1 = add_labels(m1)

# return yang menyeberangi gap (weekend / break) ditandai terpisah (lihat SA-07)
m1["gap_return"] = m1["utc"].diff() > pd.Timedelta(minutes=1)

wall_date = m1["broker_day"]
b_mask = (wall_date >= pd.Timestamp("2023-01-01")) & (wall_date <= pd.Timestamp("2023-06-30"))
c_mask = pd.Series(False, index=m1.index)
for start, end in ANOMALY_WINDOWS:
    c_mask |= (wall_date >= pd.Timestamp(start)) & (wall_date <= pd.Timestamp(end))
a_mask = ~(b_mask | c_mask)

print(f"symbol={SYMBOL} | bar={len(m1):,} | gap_return={int(m1['gap_return'].sum()):,}")
print(f"strata A={int(a_mask.sum()):,} B={int(b_mask.sum()):,} C={int(c_mask.sum()):,}")
assert int(a_mask.sum()) == 1_136_401, f"A harus 1.136.401, dapat {int(a_mask.sum())}"
assert int(b_mask.sum()) == 185_248, f"B harus 185.248, dapat {int(b_mask.sum())}"
assert int(c_mask.sum()) == 26_198, f"C harus 26.198, dapat {int(c_mask.sum())}"
assert not (b_mask & c_mask).any(), "B dan C tidak boleh tumpang tindih"
assert m1["utc"].is_monotonic_increasing, "urutan waktu harus naik"
print("PASS: stratifikasi A/B/C mereproduksi Lapisan 1; urutan naik.")


def spread_summary(frame: pd.DataFrame) -> dict[str, float]:
    """Ringkasan spread satu subset bar.

    Args:
        frame: Frame dengan kolom spread_pips.

    Returns:
        Dictionary n, p50, p95, p99, max, pct_gt_3pip.
    """
    values = frame["spread_pips"]
    return {
        "n": len(values),
        "p50": float(values.quantile(0.50)),
        "p95": float(values.quantile(0.95)),
        "p99": float(values.quantile(0.99)),
        "max": float(values.max()),
        "pct_gt_3pip": float((values > 3.0).mean()),
    }

symbol=USDJPY | bar=1,347,847 | gap_return=1,824
strata A=1,136,401 B=185,248 C=26,198
PASS: stratifikasi A/B/C mereproduksi Lapisan 1; urutan naik.


## 2. Q2 — Volatilitas per jam dinding broker

Metrik per jam broker (0–23):
- `sum_r2` = r² (realized volatility dari return kuadrat).
- `mean_abs_r` = mean|r| (robust terhadap ekor).
- `tick_volume` = proksi aktivitas.
- `spread_p95` = p95 spread (pip) pada jam itu.

Dihitung pada **strata A** (utama), dan dilaporkan versi **tanpa gap-return** sebagai
kolom terpisah. Bank M5 dipakai untuk kalibrasi M1–M4 di sel berikutnya. JANGAN membaca
baris mana pun sebagai "jam busy" tanpa melihat kolom no-gap. Catatan
Tabel ini ditampilkan penuh sebagai stream (pelajaran nb01 SA-02: `display()` hilang
saat nbconvert sehingga tabel kosong lolos sebagai "sukses").

In [4]:
"""Q2: volatilitas per jam dinding broker (strata A)."""
a_frame = m1[a_mask].copy()
a_frame["r2"] = a_frame["r"] ** 2

hourly_rows = []
for hour in range(24):
    window = a_frame[a_frame["wall_hour"] == hour]
    clean = window["r"].dropna()
    no_gap = window.loc[~window["gap_return"], "r"].dropna()
    hourly_rows.append(
        {
            "jam_broker": hour,
            "n_bar": len(window),
            "n_hari": int(window["broker_day"].nunique()),
            "sum_r2": float((clean**2).sum()),
            "mean_abs_r": float(clean.abs().mean()),
            "mean_abs_r_no_gap": float(no_gap.abs().mean()),
            "tick_volume": float(window["tick_volume"].mean()),
            "spread_p95": float(window["spread_pips"].quantile(0.95)),
        }
    )

HOURLY = pd.DataFrame(hourly_rows).set_index("jam_broker")
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:.6e}")
pd.set_option("display.max_columns", 30)
display(HOURLY)
print("TABEL Q2 per jam broker (stream, supaya tersimpan):")
print(HOURLY.to_string())

print()
print("Rasio mean|r| no-gap / mean|r| all per jam ( gap):")
ratio = HOURLY["mean_abs_r_no_gap"] / HOURLY["mean_abs_r"]
print((ratio).round(4).to_string())

print()
top6 = HOURLY["mean_abs_r"].nlargest(6).index.tolist()
print("6 jam dengan mean|r| tertinggi:", top6)

,n_bar,n_hari,sum_r2,mean_abs_r,mean_abs_r_no_gap,tick_volume,spread_p95
jam_broker,,,,,,,
0,0,0,0.000000e+00,NaN,NaN,NaN,NaN
1,49086,825,1.569929e-03,7.923998e-05,7.223943e-05,2.922224e+01,1.800000e+00
2,49441,825,1.192162e-03,9.532597e-05,9.538148e-05,5.012653e+01,1.000000e+00
3,49492,825,1.555399e-03,1.184162e-04,1.184118e-04,7.329237e+01,8.000000e-01
4,49496,825,1.368655e-03,1.037006e-04,1.037015e-04,6.192945e+01,8.000000e-01
5,49484,825,8.725898e-04,8.416740e-05,8.416052e-05,4.754676e+01,9.000000e-01
6,49469,825,9.323449e-04,7.500923e-05,7.502292e-05,4.111272e+01,9.000000e-01
7,49473,825,9.943879e-04,7.875569e-05,7.877751e-05,4.167718e+01,9.000000e-01
8,49487,825,1.013573e-03,8.864702e-05,8.865843e-05,4.911571e+01,9.000000e-01


TABEL Q2 per jam broker (stream, supaya tersimpan):
            n_bar  n_hari       sum_r2   mean_abs_r  mean_abs_r_no_gap  tick_volume   spread_p95
jam_broker                                                                                      
0               0       0 0.000000e+00          NaN                NaN          NaN          NaN
1           49086     825 1.569929e-03 7.923998e-05       7.223943e-05 2.922224e+01 1.800000e+00
2           49441     825 1.192162e-03 9.532597e-05       9.538148e-05 5.012653e+01 1.000000e+00
3           49492     825 1.555399e-03 1.184162e-04       1.184118e-04 7.329237e+01 8.000000e-01
4           49496     825 1.368655e-03 1.037006e-04       1.037015e-04 6.192945e+01 8.000000e-01
5           49484     825 8.725898e-04 8.416740e-05       8.416052e-05 4.754676e+01 9.000000e-01
6           49469     825 9.323449e-04 7.500923e-05       7.502292e-05 4.111272e+01 9.000000e-01
7           49473     825 9.943879e-04 7.875569e-05       7.877751e-05 4.16

## 3. Q3 — Volatilitas per sesi UTC (label eksklusif primer)

Label eksklusif memakai prioritas London > NewYork > Tokyo > Sydney > Off. Metrik per
sesi: `sum_r2`, `mean_abs_r` (dan no-gap), `tick_volume`, `spread_p95`, serta **rasio
volatilitas sesi terhadap rata-rata harian** (`sum_r2` sesi / `sum_r2` hari itu).

In [5]:
"""Q3: volatilitas per sesi UTC eksklusif."""
daily_r2 = a_frame.groupby("broker_day")["r2"].sum()

session_rows = []
for label in SESSION_ORDER:
    window = a_frame[a_frame["session_label"] == label]
    clean = window["r"].dropna()
    no_gap = window.loc[~window["gap_return"], "r"].dropna()
    per_day = window.groupby("broker_day")["r2"].sum()
    # CATATAN: fillna di sini BUKAN imputasi data harga. Ia hanya menyamakan indeks
    # agregat harian agar hari tanpa bar pada sesi itu menjadi 0 sumbu, bukan NaN.
    daily_total = daily_r2.reindex(per_day.index).fillna(0.0)
    share_of_daily = (per_day / daily_total.replace(0.0, np.nan)).mean()
    session_rows.append(
        {
            "sesi": label,
            "n_bar": len(window),
            "n_hari": int(window["broker_day"].nunique()),
            "sum_r2": float((clean**2).sum()),
            "mean_abs_r": float(clean.abs().mean()),
            "mean_abs_r_no_gap": float(no_gap.abs().mean()),
            "tick_volume": float(window["tick_volume"].mean()),
            "spread_p95": float(window["spread_pips"].quantile(0.95)),
            "mean_share_daily_r2": float(share_of_daily),
        }
    )

SESSIONAL = pd.DataFrame(session_rows).set_index("sesi")
pd.set_option("display.float_format", lambda v: f"{v:.6e}")
display(SESSIONAL)
print("TABEL Q3 per sesi (stream, supaya tersimpan):")
print(SESSIONAL.to_string())

active = SESSIONAL.drop(index=["Off"])
ranking = active["mean_abs_r"].sort_values(ascending=False)
print()
print("Ranking volatilitas sesi (mean|r|, primer DST-aware):")
print(ranking.round(8).to_string())

off_all = int((m1["session_label"] == "Off").sum())
print()
print(f"Bar berlabel 'Off' di SELURUH data: {off_all:,} ({off_all / len(m1):.4%})")
print(f"Bar berlabel 'Off' di strata A   : {int((a_frame['session_label'] == 'Off').sum()):,}")
print("-> 4 sesi menutup 24 jam di strata A; bar Off hanya muncul di strata B/C.")

,n_bar,n_hari,sum_r2,mean_abs_r,mean_abs_r_no_gap,tick_volume,spread_p95,mean_share_daily_r2
sesi,,,,,,,,
London,395884,825,1.332449e-02,1.107761e-04,1.107654e-04,7.824110e+01,8.000000e-01,4.612643e-01
NewYork,293134,825,4.978548e-03,7.566566e-05,7.570385e-05,5.309327e+01,1.200000e+00,1.701340e-01
Tokyo,364696,825,8.861336e-03,9.568305e-05,9.569085e-05,5.605477e+01,9.000000e-01,2.980147e-01
Sydney,82687,825,2.248997e-03,7.976286e-05,7.566798e-05,3.290303e+01,1.500000e+00,7.058695e-02
Off,0,0,0.000000e+00,NaN,NaN,NaN,NaN,NaN


TABEL Q3 per sesi (stream, supaya tersimpan):
          n_bar  n_hari       sum_r2   mean_abs_r  mean_abs_r_no_gap  tick_volume   spread_p95  mean_share_daily_r2
sesi                                                                                                               
London   395884     825 1.332449e-02 1.107761e-04       1.107654e-04 7.824110e+01 8.000000e-01         4.612643e-01
NewYork  293134     825 4.978548e-03 7.566566e-05       7.570385e-05 5.309327e+01 1.200000e+00         1.701340e-01
Tokyo    364696     825 8.861336e-03 9.568305e-05       9.569085e-05 5.605477e+01 9.000000e-01         2.980147e-01
Sydney    82687     825 2.248997e-03 7.976286e-05       7.566798e-05 3.290303e+01 1.500000e+00         7.058695e-02
Off           0       0 0.000000e+00          NaN                NaN          NaN          NaN                  NaN

Ranking volatilitas sesi (mean|r|, primer DST-aware):
sesi
London    1.107800e-04
Tokyo     9.568000e-05
Sydney    7.976000e-05
NewYork   7.5

## 4. Kalibrasi M1–M4

- **M1:** bandingkan ranking volatilitas empiris dengan prior USDJPY London > NewYork > Tokyo
  (S5-sekunder). Deviasi **dicatat sebagai temuan**, bukan dianggap salah.
- **M2:** 6 jam UTC bervolatilitas tertinggi harus berada di dalam sesi berlabel. Jika ≥2
  jatuh di `Off`, batas sesi direvisi.
- **M3:** ranking sesi harus **sepakat** antara pelabelan DST-aware (primer) dan cross-tab
  UTC-tetap. Jika berbeda → dilaporkan sebagai sensitivitas; DST-aware tetap primer.
- **M4:** sesi dengan rasio mean|r| (dalam **pip**) terhadap `spread_p95` di bawah 1 diberi
  flag deskriptif "biaya-mendominasi" sebagai bahan hipotesis — **bukan** keputusan.

**Satuan M4:** mean|r| harus dikonversi ke pip dulu: `mean|r|_pip = mean|r| * close / 0.01`.
Tidak dikonversi akan menghasilkan rasio meaningless (pelajaran satuan SA-01).

In [6]:
"""Kalibrasi M1-M4 dengan konversi satuan pip yang benar untuk M4."""
pip = 0.01
close_mean = float(a_frame["close"].mean())
unit_pip = close_mean / pip  # faktor konversi log-return -> pip

prior = ["London", "NewYork", "Tokyo"]  # S5-sekunder (prior USDJPY pip: 66/59/51)
empirical = list(ranking.index)
prior_subset = [s for s in prior if s in empirical]
empirical_subset = [s for s in empirical if s in prior]
m1_agrees = prior_subset == empirical_subset

print(f"faktor aproksimasi log-return -> pip: {unit_pip:.2f} (close_mean={close_mean:.3f})")
print("M4 memakai konversi EKSAK per bar, bukan pendekatan mean x mean.")
print(f"M1 prior (S5-sekunder): {prior_subset}")
print(f"M1 empiris            : {empirical_subset}")
print(f"M1 agree: {m1_agrees}")

xtab_rows = []
for label in SESSION_ORDER:
    window = a_frame[a_frame["session_label_xtab"] == label]
    clean = window["r"].dropna()
    xtab_rows.append({"sesi": label, "mean_abs_r_xtab": float(clean.abs().mean())})
XTAB = pd.DataFrame(xtab_rows).set_index("sesi")["mean_abs_r_xtab"]
empirical_xtab = [
    s for s in XTAB.drop(index=["Off"]).sort_values(ascending=False).index if s in prior
]
m3_agrees = empirical == empirical_xtab

print(f"M3 primer (DST-aware) : {empirical}")
print(f"M3 cross-tab (UTC)    : {empirical_xtab}")
print(f"M3 agree: {m3_agrees}  -> jika beda = sensitivitas, DST-aware tetap primer")

active_labels = [s for s in SESSION_ORDER if s != "Off"]
labels_at_top_hours = set(a_frame.loc[a_frame["utc_hour"].isin(top6), "session_label"])
off_at_top = sum(
    1 for h in top6 if a_frame.loc[a_frame["utc_hour"] == h, "session_label"].eq("Off").mean() > 0.5
)
print(f"M2 top-6 jam UTC = {top6}; label dominan jam itu: {sorted(labels_at_top_hours)}")
print(f"M2 jam dengan label Off dominan: {off_at_top} (>=2 berarti batas perlu revisi)")

m4_rows = []
for label in active_labels:
    row = SESSIONAL.loc[label]
    # Konversi EKSAK per bar (bukan mean|r| x mean harga, yang hanya pendekatan).
    per_bar_pip = (
        (np.exp(a_frame.loc[a_frame["session_label"] == label, "r"]) - 1.0)
        * a_frame.loc[a_frame["session_label"] == label, "close"]
        / pip
    )
    mean_abs_pip = float(per_bar_pip.abs().mean())
    ratio_cost = mean_abs_pip / float(row["spread_p95"]) if row["spread_p95"] > 0 else np.nan
    m4_rows.append(
        {
            "sesi": label,
            "mean_abs_r_pip": mean_abs_pip,
            "spread_p95_pip": float(row["spread_p95"]),
            "rasio_mayar": ratio_cost,
            "biaya_mendominasi": bool(ratio_cost < 1.0),
        }
    )
M4 = pd.DataFrame(m4_rows).set_index("sesi")
print()
print("M4 (mean|r| dalam pip vs spread p95):")
print(M4.round(4).to_string())
print()
print("M4 flag biaya-mendominasi (rasio < 1) - bahan hipotesis, BUKAN keputusan:")
print(M4[M4["biaya_mendominasi"]].index.tolist() or "tidak ada sesi")

faktor aproksimasi log-return -> pip: 15158.86 (close_mean=151.589)
M4 memakai konversi EKSAK per bar, bukan pendekatan mean x mean.
M1 prior (S5-sekunder): ['London', 'NewYork', 'Tokyo']
M1 empiris            : ['London', 'Tokyo', 'NewYork']
M1 agree: False


M3 primer (DST-aware) : ['London', 'Tokyo', 'Sydney', 'NewYork']
M3 cross-tab (UTC)    : ['London', 'Tokyo', 'NewYork']
M3 agree: False  -> jika beda = sensitivitas, DST-aware tetap primer
M2 top-6 jam UTC = [17, 16, 15, 3, 10, 9]; label dominan jam itu: ['London', 'NewYork', 'Tokyo']
M2 jam dengan label Off dominan: 0 (>=2 berarti batas perlu revisi)



M4 (mean|r| dalam pip vs spread p95):
         mean_abs_r_pip  spread_p95_pip  rasio_mayar  biaya_mendominasi
sesi                                                                   
London     1.665800e+00    8.000000e-01 2.082300e+00              False
NewYork    1.138600e+00    1.200000e+00 9.488000e-01               True
Tokyo      1.439300e+00    9.000000e-01 1.599200e+00              False
Sydney     1.196200e+00    1.500000e+00 7.975000e-01               True

M4 flag biaya-mendominasi (rasio < 1) - bahan hipotesis, BUKAN keputusan:
['NewYork', 'Sydney']


## 4b. TEMUAN: ekor spread 2023 adalah artefak STRATA B (bukan sifat tahun 2023)

Lapisan 1 memakai premis "2023 punya ekor spread terrible (p99 8,5 pip)" untuk memasang
asumsi `spread.by_year[2023] = 3,0`. Pengukuran di notebook ini menunjukkan premis itu
**hanya berlaku untuk strata B** (H1-2023, regime SA-03). Pada **strata A** — yang justru
dipakai analisis utama — spread 2023 normal seperti tahun lain.

Konsekuensi untuk langkah 4b: asumsi 3,0 pip untuk 2023 kemungkinan **lebih mahal sekitar 5x**
daripada yang perlu dibayar pada strata utama. Angka ini dilaporkan sebagai
**temuan terukur**, keputusan terapkan/tunda tetap milik user di 4b.

In [7]:
"""Ekor spread per strata: uji apakah "2023 buruk" adalah sifat tahun atau sifat strata B."""
strata_masks = {"A": a_mask, "B": b_mask, "C": c_mask}
print("=== spread_pips per STRATA (seluruh 2023-2026) ===")
print(f"{'strata':6} {'n':>10} {'p50':>6} {'p95':>6} {'p99':>6} {'max':>7} {'%>3pip':>9}")
strata_rows = []
for name, mask in strata_masks.items():
    values = m1.loc[mask, "spread_pips"]
    stats = spread_summary(m1[mask])
    strata_rows.append({"strata": name, **stats})
    print(
        f"{name:6} {stats['n']:>10,} {stats['p50']:>6.2f} {stats['p95']:>6.2f} "
        f"{stats['p99']:>6.2f} {stats['max']:>7.1f} {stats['pct_gt_3pip']:>9.4%}"
    )
STRATA_SPREAD = pd.DataFrame(strata_rows).set_index("strata")

print()
print("=== 2023 dipecah: H1 (strata B) vs H2 (di dalam strata A) ===")
h1_2023 = m1[b_mask]["spread_pips"]
h2_2023 = m1[(m1["utc"].dt.year == 2023) & ~b_mask & ~c_mask]["spread_pips"]
print(
    f"H1-2023 (B): p95={h1_2023.quantile(0.95):.2f} p99={h1_2023.quantile(0.99):.2f} "
    f"%>3pip={(h1_2023 > 3.0).mean():.4%}"
)
print(
    f"H2-2023 (A): p95={h2_2023.quantile(0.95):.2f} p99={h2_2023.quantile(0.99):.2f} "
    f"%>3pip={(h2_2023 > 3.0).mean():.4%}"
)

ratio_b_over_a = float(STRATA_SPREAD.loc["B", "p95"] / STRATA_SPREAD.loc["A", "p95"])
print()
print(f"Rasio p95 spread B terhadap A = {ratio_b_over_a:.2f}x")
print("-> Ekor spread 2023 mengikuti STRATA B, bukan tahun kalender.")

=== spread_pips per STRATA (seluruh 2023-2026) ===
strata          n    p50    p95    p99     max    %>3pip
A       1,136,401   0.60   1.00   1.60    55.6   0.1984%


B         185,248   1.00   5.30   8.50    34.3   5.1709%


C          26,198   0.60   0.90   1.50     8.4   0.1985%

=== 2023 dipecah: H1 (strata B) vs H2 (di dalam strata A) ===
H1-2023 (B): p95=5.30 p99=8.50 %>3pip=5.1709%
H2-2023 (A): p95=0.60 p99=1.10 %>3pip=0.2333%

Rasio p95 spread B terhadap A = 5.30x
-> Ekor spread 2023 mengikuti STRATA B, bukan tahun kalender.


## 5. Spread terukur per sesi × jam × tahun

Dari `spread_pips` M1: p50/p95/p99/maks dan % bar > 3 pip, dikelompokkan per
**tahun × sesi** dan per **tahun × jam UTC**. Tahun **2023 selalu terpisah** karena ekornya
berbeda (p99 8,5 pip menurut Lapisan 1). Ambang "3 pip" adalah **ambang deskriptif warisan
temuan fetch 2026**, bukan ambang keputusan trading.

Pengaruh ukuran: 2023 = 362.693 bar; total > 1,3 juta — pivot session × jam × tahun
disajikan per tahun agar bisa dibaca.

In [8]:
"""Spread per tahun: sesi dan jam UTC."""


def spread_summary(frame: pd.DataFrame) -> dict[str, float]:
    """Ringkasan spread satu subset bar."""
    values = frame["spread_pips"]
    return {
        "n": len(values),
        "p50": float(values.quantile(0.50)),
        "p95": float(values.quantile(0.95)),
        "p99": float(values.quantile(0.99)),
        "max": float(values.max()),
        "pct_gt_3pip": float((values > 3.0).mean()),
    }


print("=== Spread per TAHUN x SESI (eksklusif) ===")
year_session_rows = []
for year in YEARS:
    year_frame = a_frame[a_frame["utc"].dt.year == year]
    for label in SESSION_ORDER:
        stats = spread_summary(year_frame[year_frame["session_label"] == label])
        year_session_rows.append({"tahun": year, "sesi": label, **stats})
YEAR_SESSION = pd.DataFrame(year_session_rows).set_index(["tahun", "sesi"])
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
display(YEAR_SESSION)
print(YEAR_SESSION.to_string())

print()
print("=== Spread per TAHUN (seluruh data, untuk konteks ekor) ===")
year_rows = []
for year in YEARS:
    stats = spread_summary(a_frame[a_frame["utc"].dt.year == year])
    year_rows.append({"tahun": year, **stats})
YEAR_TOTAL = pd.DataFrame(year_rows).set_index("tahun")
display(YEAR_TOTAL)
print(YEAR_TOTAL.to_string())

=== Spread per TAHUN x SESI (eksklusif) ===


n    p50    p95    p99     max  pct_gt_3pip
tahun sesi                                                     
2023  London    61879 0.5000 0.5000 0.6000  3.0000       0.0000
      NewYork   45810 0.5000 0.7000 1.7000 16.0000       0.0044
      Tokyo     56760 0.5000 0.5000 0.7000  1.9000       0.0000
      Sydney    12996 0.5000 1.5000 4.0000 12.9000       0.0163
      Off           0    NaN    NaN    NaN     NaN          NaN
2024  London   122358 0.6000 1.0000 1.3000  2.4000       0.0000
      NewYork   90549 0.6000 1.3000 1.9000 55.6000       0.0031
      Tokyo    113389 0.6000 1.1000 1.4000  2.5000       0.0000
      Sydney    25136 0.6000 1.7000 2.5000  6.4000       0.0042
      Off           0    NaN    NaN    NaN     NaN          NaN
2025  London   124299 0.6000 0.8000 1.2000  4.6000       0.0003
      NewYork   91662 0.6000 1.2000 3.3000 21.7000       0.0113
      Tokyo    115299 0.6000 0.9000 1.3000  8.7000       0.0002
      Sydney    25717 0.6000 1.6000 3.0000 24.1000       0.0094
      Off           0    NaN    NaN    NaN     NaN          NaN
2026  London    87348 0.6000 0.6000 0.6000  2.1000       0.0000
      NewYork   65113 0.6000 0.7000 1.2000 16.7000       0.0015
      Tokyo     79248 0.6000 0.6000 0.7000  3.7000       0.0000
      Sydney    18838 0.6000 1.3000 1.9000  5.4000       0.0012
      Off           0    NaN    NaN    NaN     NaN          NaN

                    n    p50    p95    p99     max  pct_gt_3pip
tahun sesi                                                     
2023  London    61879 0.5000 0.5000 0.6000  3.0000       0.0000
      NewYork   45810 0.5000 0.7000 1.7000 16.0000       0.0044
      Tokyo     56760 0.5000 0.5000 0.7000  1.9000       0.0000
      Sydney    12996 0.5000 1.5000 4.0000 12.9000       0.0163
      Off           0    NaN    NaN    NaN     NaN          NaN
2024  London   122358 0.6000 1.0000 1.3000  2.4000       0.0000
      NewYork   90549 0.6000 1.3000 1.9000 55.6000       0.0031
      Tokyo    113389 0.6000 1.1000 1.4000  2.5000       0.0000
      Sydney    25136 0.6000 1.7000 2.5000  6.4000       0.0042
      Off           0    NaN    NaN    NaN     NaN          NaN
2025  London   124299 0.6000 0.8000 1.2000  4.6000       0.0003
      NewYork   91662 0.6000 1.2000 3.3000 21.7000       0.0113
      Tokyo    115299 0.6000 0.9000 1.3000  8.7000       0.0002
      Sydney    25717 0.6000 1.6000 3.00

,n,p50,p95,p99,max,pct_gt_3pip
tahun,,,,,,
2023,177445,0.5000,0.6000,1.1000,16.0000,0.0023
2024,351432,0.6000,1.2000,1.7000,55.6000,0.0011
2025,356977,0.6000,1.0000,1.7000,24.1000,0.0037
2026,250547,0.6000,0.7000,1.1000,16.7000,0.0005


            n    p50    p95    p99     max  pct_gt_3pip
tahun                                                  
2023   177445 0.5000 0.6000 1.1000 16.0000       0.0023
2024   351432 0.6000 1.2000 1.7000 55.6000       0.0011
2025   356977 0.6000 1.0000 1.7000 24.1000       0.0037
2026   250547 0.6000 0.7000 1.1000 16.7000       0.0005


In [9]:
"""Porsi bar spread > 3 pip per tahun x jam UTC (pivot 24 jam)."""
hour_share_rows = []
for year in YEARS:
    year_frame = a_frame[a_frame["utc"].dt.year == year]
    per_hour = {}
    for hour in range(24):
        window = year_frame[year_frame["utc_hour"] == hour]
        per_hour[hour] = float((window["spread_pips"] > 3.0).mean()) if len(window) else np.nan
    hour_share_rows.append(per_hour)

HOUR_SHARE = pd.DataFrame(hour_share_rows, index=list(YEARS))
HOUR_SHARE.index.name = "tahun"
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:.5f}")
display(HOUR_SHARE)
print("Porsi spread > 3 pip per tahun x jam UTC:")
print(HOUR_SHARE.to_string())

print()
print("Jam UTC dengan porsi > 3 pip tertinggi tiap tahun:")
for year in YEARS:
    top = HOUR_SHARE.loc[year].nlargest(3)
    print(f"  {year}: " + ", ".join(f"{int(h):02d}:00={v:.3%}" for h, v in top.items()))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
tahun,,,,,,,,,,,,,,,,,,,,,,,,
2023,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00039,0.02161,0.01540,0.03767,0.00156
2024,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00105,0.00425,0.00670,0.01817,0.00426,0.00413
2025,0.00148,0.00000,0.00000,0.00019,0.00000,0.00000,0.00000,0.00006,0.00000,0.00013,0.00116,0.00000,0.00000,0.00116,0.00000,0.00013,0.00026,0.00077,0.00013,0.00348,0.05537,0.02194,0.01345,0.00670
2026,0.00000,0.00000,0.00000,0.00000,0.00009,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00808,0.00370,0.00276,0.00000


Porsi spread > 3 pip per tahun x jam UTC:
           0       1       2       3       4       5       6       7       8       9       10      11      12      13      14      15      16      17      18      19      20      21      22      23
tahun                                                                                                                                                                                                
2023  0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00039 0.02161 0.01540 0.03767 0.00156
2024  0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00000 0.00105 0.00425 0.00670 0.01817 0.00426 0.00413
2025  0.00148 0.00000 0.00000 0.00019 0.00000 0.00000 0.00000 0.00006 0.00000 0.00013 0.00116 0.00000 0.00000 0.00116 0.00000 0.00013 0.00026 0.00077 0.00013 0.00348 

## 6. Fragmen YAML spread (BELUM diterapkan)

Fragmen dibangun dari **angka terukur** di atas, memakai kunci yang **sama** dengan
`configs/data.yaml` -> `spread.*`, supaya diff di langkah **4b** bersih. Angka config saat
ini (default 1,5 dan 3,0 untuk 2023) diperlakukan sebagai **ASUMSI** sampai user memutuskan
terapkan/tunda/tolak.

Fragment disimpan ke `reports/eda/2026-10-06/spread_fragment.yaml` — **bukan** ke
`configs/data.yaml`. Notebook ini **tidak pernah** menulis config.

In [10]:
"""Bangun fragmen YAML spread dari angka terukur (TIDAK menulis ke configs/)."""
mean_by_session = {
    label: float(a_frame.loc[a_frame["session_label"] == label, "spread_pips"].mean())
    for label in SESSION_ORDER
    if label != "Off"
}
# "Off" di strata A = 0 bar (empat sesi menutup 24 jam). Dicatat eksplisit
# sebagai null agar tidak disalahartikan sebagai spread tak berhingga.
mean_by_session_full = dict(mean_by_session)
mean_by_session_full["Off"] = None

fragment = {
    "_status": "PROPOSAL REVISI - BELUM DITERAPKAN. Menunggu audit Klenz (user).",
    "_sumber": "EDA-02, data/processed/M1 2023-2026, basis STRATA A (bukan campuran).",
    "_catatan": (
        "Nilai lama by_year[2023]=3,0 di config berasal dari p99 file 2023 UTUH, yang "
        "didominasi ekor STRATA B (H1-2023, regime SA-03). Di strata A, 2023 justru "
        "paling murah (mean 0,51 pip). 3,0 tidak mewakili semesta yang diperdagangkan."
    ),
    "_perbaikan_vs_fragmen_lama": (
        "Versi lama fragmen memakai p95 dan face by_year[2023]=0,6. Sekarang: biaya dasar "
        "dipindah ke cost_basis (MEAN), dan p95/p99 pindah ke stress_bands. Nilai "
        "by_year LAMA (3,0) tidak diubah di config; config lama tetap utuh."
    ),
    "_sesi": {
        "primer": "DST-aware per zona IANA (Asia/Tokyo, Europe/London, America/New_York, Australia/Sydney)",
        "jam_sesi": {
            "Tokyo": "09:00-18:00 JST (= 00:00-09:00 UTC, tanpa DST)",
            "London": "08:00-16:00 waktu lokal Europe/London",
            "NewYork": "08:00-17:00 waktu lokal America/New_York",
            "Sydney": "08:00-16:00 waktu lokal Australia/Sydney",
        },
        "label_eksklusif": "London > NewYork > Tokyo > Sydney > Off",
        "cross_tab_pembanding": "UTC-tetap: Tokyo 00-09, London 08-16, NewYork 13-22, Sydney 21-06",
        "syarat_backtest": (
            "Backtest Lapisan 5 WAJIB memakai pelabelan PRIMER yang identik "
            "(DST-aware IANA, prioritas eksklusif di atas). Cross-tab hanya untuk "
            "sensitivitas, TIDAK boleh jadi label operasi."
        ),
        "hari_perdagangan": "Tanggal broker (batas 21:00 UTC summer / 22:00 UTC winter)",
    },
    "pip_size": {"USDJPY": PIP_SIZE},
    "stress_multiplier": 2.0,
    "cost_basis": {
        "_satuan": "pip",
        "_statistik": "MEAN spread_pips strata A",
        "default": round(float(a_frame["spread_pips"].mean()), 2),
        "by_year": {
            str(year): round(
                float(a_frame.loc[a_frame["utc"].dt.year == year, "spread_pips"].mean()), 2
            )
            for year in YEARS
        },
        "by_session": {label: round(value, 2) for label, value in mean_by_session.items()},
    },
    "stress_bands": {
        "_satuan": "pip",
        "_catatan": "p95 untuk stress test; p99 untuk batas ekor. Boleh dikalikan stress_multiplier.",
        "p95": {
            "default": round(float(a_frame["spread_pips"].quantile(0.95)), 2),
            "by_year": {
                str(year): round(
                    float(
                        a_frame.loc[a_frame["utc"].dt.year == year, "spread_pips"].quantile(0.95)
                    ),
                    2,
                )
                for year in YEARS
            },
            "by_session": {
                label: round(float(SESSIONAL.loc[label, "spread_p95"]), 2)
                for label in SESSION_ORDER
                if label != "Off"
            },
        },
        "p99": {
            "default": round(float(a_frame["spread_pips"].quantile(0.99)), 2),
            "by_year": {
                str(year): round(
                    float(
                        a_frame.loc[a_frame["utc"].dt.year == year, "spread_pips"].quantile(0.99)
                    ),
                    2,
                )
                for year in YEARS
            },
        },
    },
}

yaml_text = yaml.safe_dump(fragment, sort_keys=False, allow_unicode=True)
print(yaml_text)

FRAGMENT_PATH = PROJECT_ROOT / "reports/eda/2026-10-06/spread_fragment.yaml"
FRAGMENT_PATH.parent.mkdir(parents=True, exist_ok=True)
FRAGMENT_PATH.write_text(yaml_text, encoding="utf-8")
print(f"tersimpan (bukan config): {FRAGMENT_PATH.relative_to(PROJECT_ROOT)}")

_status: PROPOSAL REVISI - BELUM DITERAPKAN. Menunggu audit Klenz (user).
_sumber: EDA-02, data/processed/M1 2023-2026, basis STRATA A (bukan campuran).
_catatan: Nilai lama by_year[2023]=3,0 di config berasal dari p99 file 2023 UTUH,
  yang didominasi ekor STRATA B (H1-2023, regime SA-03). Di strata A, 2023 justru
  paling murah (mean 0,51 pip). 3,0 tidak mewakili semesta yang diperdagangkan.
_perbaikan_vs_fragmen_lama: 'Versi lama fragmen memakai p95 dan face by_year[2023]=0,6.
  Sekarang: biaya dasar dipindah ke cost_basis (MEAN), dan p95/p99 pindah ke stress_bands.
  Nilai by_year LAMA (3,0) tidak diubah di config; config lama tetap utuh.'
_sesi:
  primer: DST-aware per zona IANA (Asia/Tokyo, Europe/London, America/New_York, Australia/Sydney)
  jam_sesi:
    Tokyo: 09:00-18:00 JST (= 00:00-09:00 UTC, tanpa DST)
    London: 08:00-16:00 waktu lokal Europe/London
    NewYork: 08:00-17:00 waktu lokal America/New_York
    Sydney: 08:00-16:00 waktu lokal Australia/Sydney
  label_eksklusi

In [11]:
"""Helper: kumpulkan semua nilai numerik dari fragmen (untuk deteksi NaN)."""


def _flatten_numbers(node):
    """Kumpulkan seluruh angka di dalam dict/list fragmen.

    Args:
        node: Struktur fragmen (dict/list/scalar).

    Yields:
        Setiap nilai numerik yang ditemukan.
    """
    if isinstance(node, dict):
        for key, value in node.items():
            if str(key).startswith("_"):
                continue
            yield from _flatten_numbers(value)
    elif isinstance(node, list):
        for item in node:
            yield from _flatten_numbers(item)
    elif isinstance(node, int | float) and not isinstance(node, bool):
        yield node

## 7. Assertion penutup + ringkasan

Semua yang diassert adalah **fakta terukur**, bukan placeholder. Tidak ada assertion
`True` hardcode (pelajaran nb01 SA-03).

In [12]:
"""Assertion penutup Q2/Q3/spread."""
checks = [
    ("Definisi sesi == Notebook 00", session_fingerprint() == EXPECTED_FINGERPRINT),
    (
        "Strata A/B/C = 1.136.401 / 185.248 / 26.198",
        (int(a_mask.sum()), int(b_mask.sum()), int(c_mask.sum())) == (1_136_401, 185_248, 26_198),
    ),
    ("Tidak ada r non-finite selain warmup", int(np.isinf(a_frame["r"].dropna()).sum()) == 0),
    ("Tabel per jam = 24 baris", len(HOURLY) == 24),
    ("Tabel per sesi = 5 baris (4 sesi + Off)", len(SESSIONAL) == 5),
    ("Jam 00-23 lengkap", list(HOURLY.index) == list(range(24))),
    (
        "Label sesi mempartisi bar strata A tanpa sisa/rumpang",
        int(SESSIONAL["n_bar"].sum()) == len(a_frame),
    ),
    (
        "Bar 'Off' di strata A = 0 (4 sesi menutup 24 jam)",
        int((a_frame["session_label"] == "Off").sum()) == 0,
    ),
    (
        "Porsi >3pip M1 berada di 0..1",
        bool(((YEAR_TOTAL["pct_gt_3pip"] >= 0) & (YEAR_TOTAL["pct_gt_3pip"] <= 1)).all()),
    ),
    (
        "2023 di strata A TIDAK lebih lebar dari 2024 (premis lama terbantah)",
        bool(YEAR_TOTAL.loc[2023, "p99"] <= YEAR_TOTAL.loc[2024, "p99"]),
    ),
    (
        "Temuan: p95 spread strata B >> strata A",
        float(STRATA_SPREAD.loc["B", "p95"]) > 3 * float(STRATA_SPREAD.loc["A", "p95"]),
    ),
    (
        "Temuan: %>3pip strata B jauh di atas A",
        float(STRATA_SPREAD.loc["B", "pct_gt_3pip"])
        > 10 * float(STRATA_SPREAD.loc["A", "pct_gt_3pip"]),
    ),
    (
        "Fragmen ditulis ke reports/, bukan configs/",
        FRAGMENT_PATH.exists() and "configs" not in str(FRAGMENT_PATH),
    ),
    ("Fragment punya cost_basis.by_year 4 tahun", len(fragment["cost_basis"]["by_year"]) == 4),
    ("Fragment punya cost_basis.by_session 4 sesi", len(fragment["cost_basis"]["by_session"]) == 4),
    (
        "Fragment punya stress_bands p95 & p99",
        "p95" in fragment["stress_bands"] and "p99" in fragment["stress_bands"],
    ),
    ("Fragment punya deklarasi _sesi + syarat backtest", "syarat_backtest" in fragment["_sesi"]),
    ("pip_size tetap dict (bukan flat)", isinstance(fragment["pip_size"], dict)),
    (
        "Fragment tidak memuat nilai NaN (tidak valid untuk config)",
        not any(isinstance(v, float) and np.isnan(v) for v in _flatten_numbers(fragment))
        and "Off" not in fragment["cost_basis"]["by_session"],
    ),
    ("M4 rasio > 0 semua sesi", bool((M4["rasio_mayar"] > 0).all())),
]

for label, passed in checks:
    print(f"{'PASS' if passed else 'GAGAL'} {label}")

failed = [label for label, passed in checks if not passed]
assert not failed, f"assertion gagal: {failed}"
print()
print(f"RINGKASAN: {len(checks) - len(failed)}/{len(checks)} assertion PASS untuk Q2/Q3/spread.")

PASS Definisi sesi == Notebook 00
PASS Strata A/B/C = 1.136.401 / 185.248 / 26.198
PASS Tidak ada r non-finite selain warmup
PASS Tabel per jam = 24 baris
PASS Tabel per sesi = 5 baris (4 sesi + Off)
PASS Jam 00-23 lengkap
PASS Label sesi mempartisi bar strata A tanpa sisa/rumpang
PASS Bar 'Off' di strata A = 0 (4 sesi menutup 24 jam)
PASS Porsi >3pip M1 berada di 0..1
PASS 2023 di strata A TIDAK lebih lebar dari 2024 (premis lama terbantah)
PASS Temuan: p95 spread strata B >> strata A
PASS Temuan: %>3pip strata B jauh di atas A
PASS Fragmen ditulis ke reports/, bukan configs/
PASS Fragment punya cost_basis.by_year 4 tahun
PASS Fragment punya cost_basis.by_session 4 sesi
PASS Fragment punya stress_bands p95 & p99
PASS Fragment punya deklarasi _sesi + syarat backtest
PASS pip_size tetap dict (bukan flat)
PASS Fragment tidak memuat nilai NaN (tidak valid untuk config)
PASS M4 rasio > 0 semua sesi

RINGKASAN: 20/20 assertion PASS untuk Q2/Q3/spread.


## Ringkasan hasil dan batasnya

**Yang dijawab:** bentuk volatilitas per jam broker dan per sesi UTC, kalibrasi M1–M4,
dan spread terukur per tahun × sesi / jam.

**Yang TIDAK dijawab dan sengaja ditolak:**
- Tidak ada kesimpulan prediktif (Q4, nb03).
- Tidak ada P&L atau keputusan metode.
- **Kalender berita di luar scope** → profil volatilitas masih memuat lonjakan berita.
  Cleaner (membersihkan) news spike adalah keputusan metode, bukan Q2.
- Fragmen spread **belum** diterapkan; menunggu 4b.

**Cara memakai angka ini nanti:** bila M4 memberi flag "biaya-mendominasi" pada suatu sesi,
itu bahan hipotesis (mis. jangan entry di jam spread lebar), **bukan** keputusan filter.
